#  Naive Bayes Classification, Decision Tree, and Artificial Neural Network — Car Price Range

- Conrad Stanley Nowowiejski - 150891
-
Martyna Rutkowska - 122196

  - Language = Python

### Goal: To determine which machine learning algorithm better classifies used cars into price categories, and to identify which vehicle attributes most strongly drive that classification.

Layer 1: The NBC/DTT report framed it as a pure model comparison — training a Naive Bayes and a Decision Tree on auction sales data to classify cars as Budget, Mid-range, or Luxury, then evaluating which model performs better and why.

Layer 2: The comparative study pushed it further with a concrete business hypothesis — that usage-based factors (odometer reading and car age) outperform subjective attributes (condition score, color, brand prestige) in predicting whether a car sells above the market average. This grounded the technical work in a real-world question relevant to dealerships and the used car industry. We pushed the boundaries by using an additional comparison model - the Artificial Neural Network, to strengthen our results. 

Use of machine learning to classify used car auction prices, compare the performance of different algorithms, and identify which vehicle characteristics are the strongest predictors of value.

##  Part 1: Setup and Data

In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier
from sklearn.tree import (DecisionTreeClassifier,
                          plot_tree, export_text)
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score,
    ConfusionMatrixDisplay, roc_curve, auc, f1_score
)
from sklearn.inspection import permutation_importance
from sklearn.preprocessing import LabelBinarizer

### 1. Load Data

In [0]:
df = pd.read_csv("car_prices.csv")
print(f"Raw dataset shape: {df.shape}")

### 2. Clean Data

In [0]:
def remove_outliers(data, column):
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1
    return data[
        (data[column] >= Q1 - 1.5 * IQR) &
        (data[column] <= Q3 + 1.5 * IQR)
    ]

df_cleaned = df.dropna().copy()
df_cleaned = remove_outliers(df_cleaned, 'sellingprice')
df_cleaned = remove_outliers(df_cleaned, 'odometer')
df_cleaned = df_cleaned[df_cleaned['sellingprice'] > 100]
print(f"Cleaned dataset shape: {df_cleaned.shape}")

### 3. Target Variable 

In [0]:
def assign_price_range(price):
    if price < 10000:   return "Budget"
    elif price < 30000: return "Mid"
    else:               return "Luxury"

df_cleaned['price_range'] = (
    df_cleaned['sellingprice'].apply(assign_price_range)
)

### 4. Feature Engineering

In [0]:
df_cleaned['car_age'] = (
    df_cleaned['year'].max() - df_cleaned['year']
)
df_cleaned['mileage_per_year'] = (
    df_cleaned['odometer'] / (df_cleaned['car_age'] + 1)
)

### 5. Encode Categorical Features

In [0]:
le = LabelEncoder()
for col in ['make', 'body', 'color', 'transmission']:
    top_10 = df_cleaned[col].value_counts().index[:10]
    df_cleaned[col] = df_cleaned[col].where(
        df_cleaned[col].isin(top_10), 'Other'
    )
    df_cleaned[col + '_enc'] = le.fit_transform(
        df_cleaned[col].astype(str)
    )

### 6. Define Features and Target

In [0]:
features = [
    'year', 'car_age', 'condition', 'odometer', 'mileage_per_year',
    'mmr', 'make_enc', 'body_enc', 'transmission_enc', 'color_enc'
]

X = df_cleaned[features]
y = df_cleaned['price_range']

### 7. Train/Test Split

In [0]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\n✓ Data ready")
print(f"✓ X_train: {X_train.shape} | X_test: {X_test.shape}")
print(f"✓ Classes: {y.unique().tolist()}")

## Part 2: NBC vs Decision Tree

### 8. Pre-Modeling 

### 8.1. Class Distribution 

In [0]:
plt.figure(figsize=(6, 4))
df_cleaned['price_range'].value_counts().plot(
    kind='bar', color=['#4C72B0', '#DD8452', '#55A868']
)
plt.title("Price Range Class Distribution")
plt.xlabel("Price Range")
plt.ylabel("Count")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig("class_distribution.png", dpi=150)
plt.show()


### 8.2. Correlation Heatmap 

In [0]:
plt.figure(figsize=(10, 8))
corr_cols = ['sellingprice', 'odometer', 'mmr', 'condition', 'year']
correlation_matrix = df_cleaned[corr_cols].corr()
sns.heatmap(
    correlation_matrix, annot=True, cmap='coolwarm', fmt='.2f'
)
plt.title("Feature Correlation Heatmap")
plt.tight_layout()
plt.savefig("correlation_heatmap.png", dpi=150)
plt.show()

### 8.3. Depreciation Curve 

In [0]:
plt.figure(figsize=(10, 6))
sns.lineplot(data=df_cleaned, x='car_age', y='sellingprice')
plt.title("Depreciation Curve: Average Price by Car Age")
plt.xlabel("Car Age (years)")
plt.ylabel("Average Selling Price ($)")
plt.grid(True)
plt.tight_layout()
plt.savefig("depreciation_curve.png", dpi=150)
plt.show()

### 8.4. Depreciation Rate Per Year

In [0]:
depreciation_df = df_cleaned.groupby(
    'car_age')['sellingprice'].mean().reset_index()
depreciation_df.columns = ['car_age', 'avg_price']
depreciation_df['price_drop'] = (
    depreciation_df['avg_price'].diff(-1)
)
depreciation_df['depreciation_rate_%'] = (
    depreciation_df['price_drop'] /
    depreciation_df['avg_price'] * 100
).round(2)

avg_annual_rate = depreciation_df['depreciation_rate_%'].mean()

print("=== Depreciation Analysis ===")
print(depreciation_df[
    ['car_age', 'avg_price', 'price_drop', 'depreciation_rate_%']
].head(15).to_string(index=False))
print(f"\nAverage Annual Rate: {avg_annual_rate:.2f}%")
print(
    f"Steepest Drop: Year "
    f"{depreciation_df['depreciation_rate_%'].idxmax()} "
    f"({depreciation_df['depreciation_rate_%'].max():.2f}%)"
)
print(
    f"Slowest Drop: Year "
    f"{depreciation_df['depreciation_rate_%'].idxmin()} "
    f"({depreciation_df['depreciation_rate_%'].min():.2f}%)"
)

plt.figure(figsize=(12, 5))
plt.bar(
    depreciation_df['car_age'].iloc[:-1],
    depreciation_df['depreciation_rate_%'].iloc[:-1],
    color='#DD8452', edgecolor='white'
)
plt.axhline(
    y=avg_annual_rate, color='red', linestyle='--',
    label=f'Average: {avg_annual_rate:.2f}%'
)
plt.title("Annual Depreciation Rate by Car Age")
plt.xlabel("Car Age (years)")
plt.ylabel("Depreciation Rate (%)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("depreciation_rate.png", dpi=150)
plt.show()

### 9. Train NBC (no scaling)

In [0]:
gnb = GaussianNB(var_smoothing=1e-9)
gnb.fit(X_train, y_train)
y_pred_nb = gnb.predict(X_test)

print("=== Naive Bayes Results ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred_nb):.4f}")
print(classification_report(y_test, y_pred_nb))

### 10. Train Decision Tree (no sclaing)

In [0]:
dtree = DecisionTreeClassifier(
    criterion='gini',
    max_depth=5,
    min_samples_split=50,
    min_samples_leaf=20,
    random_state=42
)
dtree.fit(X_train, y_train)
y_pred_dt = dtree.predict(X_test)

print("=== Decision Tree Results ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred_dt):.4f}")
print(classification_report(y_test, y_pred_dt))

### 11. NBC vs Decision Tree — Confusion Matrices 

In [0]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ConfusionMatrixDisplay(
    confusion_matrix(y_test, y_pred_nb),
    display_labels=gnb.classes_
).plot(ax=axes[0], colorbar=False)
axes[0].set_title("Naive Bayes — Confusion Matrix")

ConfusionMatrixDisplay(
    confusion_matrix(y_test, y_pred_dt),
    display_labels=dtree.classes_
).plot(ax=axes[1], colorbar=False)
axes[1].set_title("Decision Tree — Confusion Matrix")

plt.tight_layout()
plt.savefig("nbc_dt_confusion_matrices.png", dpi=150)
plt.show()

### 12. NBC vs Decision Tree — Accuracy Comparison

In [0]:
models_primary = ['Naive Bayes', 'Decision Tree']
accuracies_primary = [
    accuracy_score(y_test, y_pred_nb),
    accuracy_score(y_test, y_pred_dt)
]

plt.figure(figsize=(7, 4))
bars = plt.bar(
    models_primary, accuracies_primary,
    color=['#4C72B0', '#55A868'], width=0.4
)
plt.ylim(0, 1.05)
plt.ylabel("Accuracy")
plt.title("NBC vs Decision Tree — Accuracy Comparison")
for bar, acc in zip(bars, accuracies_primary):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.01,
        f"{acc:.4f}", ha='center', fontsize=11
    )
plt.tight_layout()
plt.savefig("nbc_dt_accuracy.png", dpi=150)
plt.show()

### 13. Decision Tree — Tree Visualisation

In [0]:
plt.figure(figsize=(22, 10))
plot_tree(
    dtree,
    feature_names=features,
    class_names=dtree.classes_,
    filled=True,
    fontsize=9,
    max_depth=2,
    impurity=True,
    proportion=False
)
plt.title("Decision Tree Logic — Top 2 Levels (Full depth=5)")
plt.tight_layout()
plt.savefig("decision_tree_plot.png", dpi=150)
plt.show()

### 14: Decision Tree — Text Rules (Decision Tree only, NBC is not possible)

In [0]:
rules = export_text(dtree, feature_names=features, max_depth=3)
print("=== Decision Tree Rules (Top 3 Levels) ===")
print(rules)

### 15: NBC vs Decision Tree — Feature Importance

In [0]:
dt_importance = pd.DataFrame({
    'Feature':    features,
    'Importance': dtree.feature_importances_
}).sort_values('Importance', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(
    x='Importance', y='Feature', data=dt_importance,
    hue='Feature', palette='viridis', legend=False
)
plt.title("Decision Tree — Gini Feature Importance")
plt.xlabel("Importance Score")
plt.ylabel("Feature")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("dt_feature_importance.png", dpi=150)
plt.show()

print("\nFeature Importance Ranking:")
print(dt_importance.to_string(index=False))

### 16. Decision Tree — Depth Sweep

In [0]:
train_scores, test_scores = [], []
depths = range(1, 20)

for d in depths:
    dt_temp = DecisionTreeClassifier(max_depth=d, random_state=42)
    dt_temp.fit(X_train, y_train)
    train_scores.append(
        accuracy_score(y_train, dt_temp.predict(X_train))
    )
    test_scores.append(
        accuracy_score(y_test, dt_temp.predict(X_test))
    )

plt.figure(figsize=(12, 5))
plt.plot(depths, train_scores,
         label='Training Accuracy', color='#DD8452', marker='o')
plt.plot(depths, test_scores,
         label='Test Accuracy', color='#4C72B0', marker='o')
plt.axvline(
    x=5, color='red', linestyle='--', label='Current max_depth=5'
)
plt.title("Decision Tree — Accuracy vs Tree Depth")
plt.xlabel("max_depth")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("depth_sweep.png", dpi=150)
plt.show()

### 17. NBC vs Decision Tree — ROC Curves

In [0]:
lb = LabelBinarizer()
y_test_bin = lb.fit_transform(y_test)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
models_roc_primary = [
    (gnb,   X_test, 'Naive Bayes',   '#4C72B0'),
    (dtree, X_test, 'Decision Tree', '#55A868'),
]

for ax, (model, x_data, label, color) in zip(
    axes, models_roc_primary
):
    y_score = model.predict_proba(x_data)
    for i, cls in enumerate(model.classes_):
        fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_score[:, i])
        roc_auc = auc(fpr, tpr)
        ax.plot(fpr, tpr, label=f'{cls} (AUC={roc_auc:.3f})')
    ax.plot([0, 1], [0, 1], 'k--', alpha=0.4)
    ax.set_title(f'{label} — ROC Curves')
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("nbc_dt_roc.png", dpi=150)
plt.show()

### 18: NBC vs Decision Tree — Primary Summary

In [0]:
results_primary = pd.DataFrame({
    'Model': ['Naive Bayes', 'Decision Tree'],
    'Accuracy': [
        accuracy_score(y_test, y_pred_nb),
        accuracy_score(y_test, y_pred_dt)
    ],
    'Macro F1': [
        f1_score(y_test, y_pred_nb,  average='macro'),
        f1_score(y_test, y_pred_dt,  average='macro')
    ]
})

print("=== NBC vs Decision Tree — Primary Comparison ===")
print(results_primary.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].bar(
    results_primary['Model'], results_primary['Accuracy'],
    color=['#4C72B0', '#55A868'], width=0.4
)
axes[0].set_ylim(0, 1.05)
axes[0].set_title("Accuracy — NBC vs Decision Tree")
axes[0].set_ylabel("Accuracy")
for i, v in enumerate(results_primary['Accuracy']):
    axes[0].text(i, v + 0.01, f"{v:.4f}", ha='center', fontsize=10)

axes[1].bar(
    results_primary['Model'], results_primary['Macro F1'],
    color=['#4C72B0', '#55A868'], width=0.4
)
axes[1].set_ylim(0, 1.05)
axes[1].set_title("Macro F1 — NBC vs Decision Tree")
axes[1].set_ylabel("Macro F1 Score")
for i, v in enumerate(results_primary['Macro F1']):
    axes[1].text(i, v + 0.01, f"{v:.4f}", ha='center', fontsize=10)

plt.tight_layout()
plt.savefig("nbc_dt_summary.png", dpi=150)
plt.show()

## Part 3: ANN — Standalone Deep Dive

The ANN is a more complex model included as a performance benchmark and to demonstrate what accuracy is achievable when interpretability is not a constraint.

### 19. Scale Features for ANN

In [0]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print("✓ Features scaled for ANN")
print(f"✓ Mean (first feature): {X_train_scaled[:, 0].mean():.4f}")
print(f"✓ Std  (first feature): {X_train_scaled[:, 0].std():.4f}")

### 20. Train ANN

In [0]:
mlp = MLPClassifier(
    hidden_layer_sizes=(128, 64),
    activation='relu',
    solver='adam',
    alpha=0.001,
    learning_rate_init=0.001,
    max_iter=300,
    random_state=42
)
mlp.fit(X_train_scaled, y_train)
y_pred_mlp = mlp.predict(X_test_scaled)

print("=== ANN (MLP) Results ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred_mlp):.4f}")
print(classification_report(y_test, y_pred_mlp))


### 21. ANN — Confusion Matrix

In [0]:
fig, axes = plt.subplots(1, 1, figsize=(7, 5))
ConfusionMatrixDisplay(
    confusion_matrix(y_test, y_pred_mlp),
    display_labels=mlp.classes_
).plot(colorbar=False)
plt.title("ANN (MLP) — Confusion Matrix")
plt.tight_layout()
plt.savefig("ann_confusion_matrix.png", dpi=150)
plt.show()

### 22. ANN — ROC Curves

In [0]:
fig, ax = plt.subplots(figsize=(8, 6))
y_score_mlp = mlp.predict_proba(X_test_scaled)

for i, cls in enumerate(mlp.classes_):
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_score_mlp[:, i])
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, label=f'{cls} (AUC={roc_auc:.3f})')

ax.plot([0, 1], [0, 1], 'k--', alpha=0.4)
ax.set_title("ANN (MLP) — ROC Curves Per Class")
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("ann_roc.png", dpi=150)
plt.show()

### 23. ANN — Permutation Feature Importance

Uses binary target for importance analysis. Separate _b variables protect the original split

In [0]:
median_price = df_cleaned['sellingprice'].median()
y_binary = (df_cleaned['sellingprice'] > median_price).astype(int)

X_b = df_cleaned[features]
X_train_b, X_test_b, y_train_b, y_test_b = train_test_split(
    X_b, y_binary, test_size=0.2, random_state=42
)
X_train_b_scaled = scaler.fit_transform(X_train_b)
X_test_b_scaled  = scaler.transform(X_test_b)

mlp_binary = MLPClassifier(
    hidden_layer_sizes=(128, 64), activation='relu',
    solver='adam', alpha=0.001, max_iter=300, random_state=42
)
mlp_binary.fit(X_train_b_scaled, y_train_b)

nn_perm = permutation_importance(
    mlp_binary, X_test_b_scaled, y_test_b,
    n_repeats=5, random_state=42
)
nn_importance = pd.DataFrame({
    'Feature':    features,
    'Importance': nn_perm.importances_mean
}).sort_values('Importance', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(
    x='Importance', y='Feature', data=nn_importance,
    hue='Feature', palette='plasma', legend=False
)
plt.title("ANN — Permutation Feature Importance")
plt.xlabel("Importance Score")
plt.ylabel("Feature")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("ann_feature_importance.png", dpi=150)
plt.show()

print("\nANN — Top 5 Features:")
print(nn_importance.head(5).to_string(index=False))

### 24. ANN — Binary ROC vs NBC and DT

In [0]:
dtree_b = DecisionTreeClassifier(max_depth=5, random_state=42)
dtree_b.fit(X_train_b, y_train_b)

gnb_b = GaussianNB(var_smoothing=1e-9)
gnb_b.fit(X_train_b, y_train_b)

plt.figure(figsize=(10, 6))
for model, x_data, label, color in [
    (mlp_binary, X_test_b_scaled, 'ANN (MLP)',    'purple'),
    (dtree_b,    X_test_b,        'Decision Tree', '#55A868'),
    (gnb_b,      X_test_b,        'Naive Bayes',   '#4C72B0'),
]:
    probs = model.predict_proba(x_data)[:, 1]
    fpr, tpr, _ = roc_curve(y_test_b, probs)
    roc_auc = auc(fpr, tpr)
    plt.plot(
        fpr, tpr,
        label=f'{label} (AUC = {roc_auc:.3f})', color=color
    )

plt.plot([0, 1], [0, 1], 'k--', label='Random Baseline (AUC=0.500)')
plt.title('ROC Comparison — All Three Models (Binary Task)')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("all_models_roc_binary.png", dpi=150)
plt.show()

## Part 4: Final Three-Model Summary

### 25. Three-Model Final Comparison

In [0]:
results_final = pd.DataFrame({
    'Model': ['Naive Bayes', 'Decision Tree', 'ANN (MLP)'],
    'Accuracy': [
        accuracy_score(y_test, y_pred_nb),
        accuracy_score(y_test, y_pred_dt),
        accuracy_score(y_test, y_pred_mlp)
    ],
    'Macro F1': [
        f1_score(y_test, y_pred_nb,  average='macro'),
        f1_score(y_test, y_pred_dt,  average='macro'),
        f1_score(y_test, y_pred_mlp, average='macro')
    ]
})

print("=== Final Three-Model Comparison ===")
print(results_final.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(
    results_final['Model'], results_final['Accuracy'],
    color=['#4C72B0', '#55A868', '#DD8452'], width=0.4
)
axes[0].set_ylim(0, 1.05)
axes[0].set_title("Accuracy — All Three Models")
axes[0].set_ylabel("Accuracy")
for i, v in enumerate(results_final['Accuracy']):
    axes[0].text(
        i, v + 0.01, f"{v:.4f}", ha='center', fontsize=10
    )

axes[1].bar(
    results_final['Model'], results_final['Macro F1'],
    color=['#4C72B0', '#55A868', '#DD8452'], width=0.4
)
axes[1].set_ylim(0, 1.05)
axes[1].set_title("Macro F1 — All Three Models")
axes[1].set_ylabel("Macro F1 Score")
for i, v in enumerate(results_final['Macro F1']):
    axes[1].text(
        i, v + 0.01, f"{v:.4f}", ha='center', fontsize=10
    )

plt.tight_layout()
plt.savefig("final_three_model_comparison.png", dpi=150)
plt.show()

### 26. Feature Importance — All Three Models

In [0]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.barplot(
    x='Importance', y='Feature', data=dt_importance,
    ax=axes[0], hue='Feature', palette='viridis', legend=False
)
axes[0].set_title('Decision Tree — Gini Importance')

sns.barplot(
    x='Importance', y='Feature', data=nn_importance,
    ax=axes[1], hue='Feature', palette='plasma', legend=False
)
axes[1].set_title('ANN — Permutation Importance')

plt.suptitle(
    "Feature Importance Comparison — Decision Tree vs ANN",
    fontsize=13, fontweight='bold', y=1.02
)
plt.tight_layout()
plt.savefig("importance_comparison.png", dpi=150)
plt.show()

print("\nDecision Tree vs ANN — Feature Agreement:")
merged = dt_importance.rename(
    columns={'Importance': 'DT_Importance'}
).merge(
    nn_importance.rename(columns={'Importance': 'ANN_Importance'}),
    on='Feature'
)
print(merged.to_string(index=False))